In [1]:
from crewai import Agent, Task, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
from utils import get_openai_api_key, get_exa_api_key, load_env
from IPython.display import Markdown
import yaml

# Load .env first — sets OPENAI_API_BASE, OPENAI_BASE_URL, OPENAI_MODEL_NAME
load_env()

# set up the OpenAI API key
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the EXA API key
os.environ["EXA_API_KEY"] = get_exa_api_key()


Using Custom API Base: https://api.deepseek.com
Using Model: deepseek-v4-flash
Using Custom API Base: https://api.deepseek.com
Using Model: deepseek-v4-flash
Using Custom API Base: https://api.deepseek.com
Using Model: deepseek-v4-flash


In [2]:
# create the tool instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL")) 
print("... ",os.getenv("EXA_BASE_URL"))
scrape_website_tool = ScrapeWebsiteTool()

# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)

# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=30, # 150
        max_iter=15  # 15
        )

...  None


In [3]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    ### START CODE HERE ###

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower):
        return (False,
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )

    ### END CODE HERE ###
    return (True, output)


In [4]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [5]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

In [6]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

### START CODE HERE ###

# create the tasks using the configuration
create_research_plan = Task(
    config=task_config['create_research_plan'],
    agent=research_planner
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task(
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail],
)

### END CODE HERE ###

In [7]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

In [8]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],

    ### START CODE HERE ###
    
    # add memory to the crew
    # memory=True,
    memory=False, # set to True to enable memory across tasks, False to disable memory
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]

    ### END CODE HERE ###
)

In [9]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
    "user_query": "Evaluate the top one emerging AI tool for automating competitive market analysis, including its features, limitations, costs, and ideal use cases for a mid-sized marketing firm."
}

### END CODE HERE ###

In [10]:
# Execute the crew's tasks
result = deep_research_crew.kickoff(inputs=inputs)

/Users/taozh/Documents/agents/agent-jupyter/venv/lib/python3.13/site-packages/pydantic/main.py:250: UserWarning: method callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top one emerging AI tool for automating competitive market   │
│  analysis, including its features, limitations, costs, and ideal use cases for a mid-sized marketing firm."     │
│  into specific topics and key questions that need investigation. Create a focused research plan.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Research Plan: Evaluating the Top Emerging AI Tool for Automating Competitive Market Analysis               │
│                                                                                                                 │
│  ### 1. Main Research Topics to Investigate                                                                     │
│                                                                                                                 │
│  - **Topic A: Identification of the “Top” Emerging AI Tool**                                                    │
│    Establish objective criteria to determine which AI tool is currently considered the leading emerging         │
│  solution for automated competitive market analysis, and identify that tool by name.                            │
│                                                                                                                 │
│  - **Topic B: Features and Capabilities**                                                                       │
│    Detail the core functionalities, unique selling points, and automation aspects of the identified tool.       │
│                                                                                                                 │
│  - **Topic C: Limitations and Constraints**                                                                     │
│    Uncover the tool’s known weaknesses, technical limitations, data quality issues, or scenarios where it       │
│  underperforms.                                                                                                 │
│                                                                                                                 │
│  - **Topic D: Cost Structure and Pricing Models**                                                               │
│    Analyze the pricing tiers, subscription plans, hidden fees, and overall affordability for a mid‑sized        │
│  marketing firm.                                                                                                │
│                                                                                                                 │
│  - **Topic E: Ideal Use Cases for a Mid‑Sized Marketing Firm**                                                  │
│    Specify the firm profiles (e.g., team size, industry vertical, data complexity, strategic goals) where this  │
│  tool delivers the highest value, and describe real‑world application examples.                                 │
│                                                                                                                 │
│  - **Topic F: Competitive Landscape & Verification**                                                            │
│    Briefly assess how this tool compares to its closest competitors to confirm its “top” status, and verify     │
│  its emerging‑market reputation through recent reviews, analyst reports, and user feedback.                     │
│                                                                                                                 │
│  ### 2. Key Questions for Each Topic                                                                            │
│                                                                                                                 │
│  **Topic A – Identification**                          

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool exa_search_tool executed with result: Title: IntelCue | AI Competitive Intelligence Platform & Market Monitoring
URL: https://www.intelcue.ai/
ID: https://www.intelcue.ai/
Score: None
Published Date: 2024-11-15T00:00:00.000Z
Author: None
...
Tool exa_search_tool executed with result: Title: Market Guide for Competitive and Market Intelligence Tools
URL: https://www.gartner.com/en/documents/5487795
ID: https://www.gartner.com/en/documents/5487795
Score: None
Published Date: 2024-06...
Tool exa_search_tool executed with result: Title: Steve vs. Kompyte: How to Choose an AI‑Native Competitive‑Intelligence Platform in 2025 | by Taka Morinaga | Medium
URL: https://medium.com/@taka_aifounder/steve-vs-kompyte-how-to-choose-an-ai-...
Tool exa_search_tool executed with result: Error executing tool: HTTPSConnectionPool(host='api.exa.ai', port=443): Max retries exceeded with url: /search (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in v...
Tool exa_sea

...

Tool read_website_content executed with result: The following text is scraped website content:
The AI Competitive Intelligence Platform
Platform Competitive Intelligence Competitors Suppliers Customers Battlecards Monitoring & Signals News and Web ...
Tool read_website_content executed with result: The following text is scraped website content:
About Us | AeraVision
Platform Competitive Intelligence Competitors Suppliers Customers Battlecards Monitoring & Signals News and Web Social Media Websit...
Tool read_website_content executed with result: The following text is scraped website content:
Plans & Pricing | AeraVision
Platform Competitive Intelligence Competitors Suppliers Customers Battlecards Monitoring & Signals News and Web Social Media...
Tool read_website_content executed with result: The following text is scraped website content:
Top 5 AI Tools For Competitor Analysis - A Comprehensive Review | AeraVision
Platform Competitive Intelligence Competitors Suppliers Customers Battle

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # QUALITY ASSESSMENT REPORT                                                                                    │
│  ## Evaluating Research Data for "Top Emerging AI Tool for Automating Competitive Market Analysis"              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## ⚠️ EXECUTIVE SUMMARY: APPROVAL CONDITIONAL — REQUIRES CORRECTIONS                                           │
│                                                                                                                 │
│  **Verdict: APPROVED WITH SIGNIFICANT REVISIONS REQUIRED.** The research identifies a real product              │
│  (AeraVision) with largely accurate feature and pricing data, but contains several critical factual             │
│  misrepresentations, sourcing biases, and unverifiable claims that must be corrected before final report        │
│  generation. Do not proceed to final report without addressing the issues flagged below.                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1) ORIGINAL DATA INVENTORY                                                                                  │
│                                                                                                                 │
│  ### Source Type Breakdown                                                                                      │
│                                                                                                                 │
│  | Source Type | Count | Examples |                                                                             │
│  |-------------|-------|---------|                                                                              │
│  | **Official Product Website** | 7+ pages | aeravision.com (home, about, pricing, features, etc.) |            │
│  | **AeraVision Self-Published Blog Posts** | 7+ posts | "Top 5 AI Tools", "Kompyte vs AeraVision", "Crayon vs  │
│  AeraVision", "Cision vs AeraVision", "Vocus Alternative", "Newslever Alternative", "Tired of Legacy Tools" |   │
│  | **Analyst Reports (Verified)** | 2 | Forrester Wave Q4 2024 (PDF), Gartner Market Guide June 2024 |          │
│  | **Third-Party Listings/Directories** | 2 | Toolsify.ai directory, G2 Summer 2024 Report |                    │
│  | **Third-Party Comparison Blogs** | 3 | Inflowave.io, Raven Seer, RivalSense |                                │
│  | **Competitor Official Sources** | 2 | Valona Intelligence, Klue websites |                                   │
│  | **Industry Consultant Reports** | 1 | Pedowitz Group |                                                       │
│                                                                                                                 │
│  ---                                                   

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query "Evaluate the top one emerging AI tool     │
│  for automating competitive market analysis, including its features, limitations, costs, and ideal use cases    │
│  for a mid-sized marketing firm." using all verified research data. Structure it with clear sections, include   │
│  citations, and provide actionable insights.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # FINAL RESEARCH REPORT                                                                                        │
│                                                                                                                 │
│  ## Evaluating the Top Emerging AI Tool for Automating Competitive Market Analysis                              │
│                                                                                                                 │
│  **Prepared for:** Decision‑makers at a Mid‑Sized Marketing Firm                                                │
│  **Date:** April 2025                                                                                           │
│  **Researcher:** Report Writer (based on verified data)                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│                                                                                                                 │
│  This report evaluates the top emerging AI tool for automating competitive market analysis, focusing on the     │
│  needs of a mid‑sized marketing firm (10–50 employees). The identified tool is **AeraVision**, a cloud‑based    │
│  competitive intelligence platform built on an AI‑native agent framework. AeraVision offers transparent         │
│  pricing ($0–$399/month), autonomous signal detection across 100,000+ sources, an AI assistant (AeraAI),        │
│  automated battlecard generation, and integrations with common marketing tools.                                 │
│                                                                                                                 │
│  **Critical caveats:**                                                                                          │
│  - No independent analyst report (e.g., Forrester, Gartner) has yet recognised AeraVision as a market leader.   │
│  The Forrester Wave™ Q4 2024 evaluated 11 enterprise vendors but **did not include AeraVision**.                │
│  - Performance claims (e.g., “95% satisfaction”, “30,000+ customers”, “3x ROI”) are self‑reported by            │
│  AeraVision and not independently verified.                                                                     │
│  - No verified user reviews from G2, Capterra, or TrustRadius were found at the time of research.               │
│                                                                                                                 │
│  Nevertheless, for mid‑sized marketing firms seeking an affordable, easy‑to‑deploy AI solution that covers      │
│  competitor, customer, and supplier intelligence, AeraVision presents a compelling value proposition when its   │
│  claims are taken with appropriate caution. This report provides a balanced, evidence‑based assessment to       │
│  support your decision.                                                                                         │
│                                                        

Report successfully saved to: research_report.md
